In [ ]:
# ---
# title: FlavorMap
# description: The ultimate global flavor mapping and prediction system.
# show-code: false
# ---


In [ ]:
# ---
# title: FlavorMap
# description: The ultimate global flavor mapping and prediction system.
# show-code: false
# ---

import json
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# --- Persistence (Updated for Mercury/Local) ---
# This will save the data in the same folder as your notebook
BASE_PATH = "./"
IMG_FOLDER = os.path.join(BASE_PATH, "food_images")
DATA_FILE = os.path.join(BASE_PATH, "flavormap_data.json")

if not os.path.exists(IMG_FOLDER):
    os.makedirs(IMG_FOLDER)

def save_data():
    with open(DATA_FILE, 'w') as f:
        json.dump(db, f, indent=4)

def load_data():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, 'r') as f:
            data = json.load(f)
            if "pending_foods" not in data: data["pending_foods"] = []
            return data
    # Default Starting State
    return {
        "users": [{"username": "HenndralSaige", "password": "master69", "traits": [2,2,2,2,2,2,2]}],
        "foods": [],
        "reviews": [],
        "pending_foods": []
    }

db = load_data()
current_user = None
traits_list = ["Sweet", "Sour", "Salty", "Bitter", "Umami", "Spicy", "Richness"]

# --- Math Logic ---
def predict_score(food, target_user):
    f_revs = [r for r in db['reviews'] if r['food'] == food['name']]
    if not f_revs: return "No Global Data"
    avg_r = sum(r['rating'] for r in f_revs) / len(f_revs)
    t_sums = [0]*7
    for r in f_revs:
        rev_obj = next((u for u in db['users'] if u['username'] == r['user']), None)
        if rev_obj:
            for i in range(7): t_sums[i] += rev_obj['traits'][i]
    avg_vec = [s / len(f_revs) for s in t_sums]

    def compute_raw(F, P): return sum(f * p for f, p in zip(F, P))
    def compute_max(F): return sum(f * 2 for f in F)
    def normalize(raw, m): return 5 * (raw / m) if m != 0 else 0

    max_v = compute_max(food['traits'])
    exp_avg = normalize(compute_raw(food['traits'], avg_vec), max_v)
    err = avg_r - exp_avg
    base_t = normalize(compute_raw(food['traits'], target_user['traits']), max_v)
    return round(max(-5, min(5, base_t + err)), 2)

# --- GUI Components ---
out = widgets.Output()

def refresh_ui():
    clear_output(wait=True)
    if current_user is None: show_login()
    else: show_main_menu()

def show_login():
    with out:
        clear_output()
        print("=== FLAVORMAP LOGIN ===")
        un = widgets.Text(description="Username:")
        pw = widgets.Password(description="Password:")
        status = widgets.Label()
        btn_l = widgets.Button(description="Login", button_style='primary')
        btn_r = widgets.Button(description="Register", button_style='success')

        def log_act(b=None):
            global current_user
            u = next((u for u in db['users'] if u['username'] == un.value.strip() and u['password'] == pw.value.strip()), None)
            if u: current_user = u; refresh_ui()
            else: status.value = "❌ Invalid Credentials."

        def reg_act(b):
            if any(u['username'] == un.value.strip() for u in db['users']): status.value = "❌ Taken."
            else:
                new_u = {"username": un.value.strip(), "password": pw.value.strip(), "traits": [0]*7}
                db['users'].append(new_u); save_data(); current_user = new_u; refresh_ui()

        un.on_submit(log_act); pw.on_submit(log_act); btn_l.on_click(log_act); btn_r.on_click(reg_act)
        display(un, pw, status, widgets.HBox([btn_l, btn_r]))
    display(out)

def show_main_menu():
    with out:
        clear_output()
        food_options = [f['name'] for f in db['foods']]
        is_admin = (current_user['username'] == "HenndralSaige")

        # TAB 1: DASHBOARD
        search = widgets.Text(placeholder='Find a flavor...', description='Search:')
        d_out = widgets.Output()
        def upd_l(change=None):
            with d_out:
                clear_output()
                q = search.value.lower()
                sf = sorted(db['foods'], key=lambda x: x['name'].upper())
                cl = ""
                for f in sf:
                    if q in f['name'].lower():
                        if f['name'][0].upper() != cl: cl = f['name'][0].upper(); print(f"\n--- {cl} ---")
                        print(f" • {f['name']}")
        search.observe(upd_l, 'value'); upd_l()
        tab1 = widgets.VBox([search, d_out])

        # TAB 2: FOOD LAB
        fn = widgets.Text(description="Food Name:")
        sls = [widgets.IntSlider(value=0, min=0, max=10, description=t) for t in traits_list]
        up = widgets.FileUpload(accept='image/*', multiple=False, description="Upload Image")
        btn_f = widgets.Button(description="Submit Food", button_style='info')
        msg_f = widgets.Label()

        def add_f(b):
            if not fn.value.strip(): return
            img = ""
            if up.value:
                k = list(up.value.keys())[0]; cont = up.value[k]['content']
                img = f"{fn.value.strip().replace(' ','_')}_{k}"
                with open(os.path.join(IMG_FOLDER, img), 'wb') as f: f.write(cont)

            entry = {"name": fn.value.strip(), "traits": [s.value for s in sls], "image": img, "by": current_user['username']}
            if is_admin:
                db['foods'].append(entry); msg_f.value = "✅ Food Added Directly!"
            else:
                msg_f.value = "⏳ Sent to Admin for Verification."
                db['pending_foods'].append(entry)
            save_data(); fn.value = ""

        btn_f.on_click(add_f)

        rev_f = widgets.Dropdown(options=food_options if food_options else ["No Foods"], description="Select Food:")
        rev_s = widgets.FloatSlider(value=0, min=-5, max=5, step=0.1, description="Rating:")
        btn_r = widgets.Button(description="Save Review", button_style='success')
        def add_r(b):
            if rev_f.value == "No Foods": return
            db['reviews'] = [r for r in db['reviews'] if not (r['user']==current_user['username'] and r['food']==rev_f.value)]
            db['reviews'].append({"user": current_user['username'], "food": rev_f.value, "rating": rev_s.value})
            save_data(); print(f"✅ Rated {rev_f.value}")
        btn_r.on_click(add_r)
        tab2 = widgets.VBox([widgets.Label("SUBMIT NEW FOOD"), fn] + sls + [up, btn_f, msg_f, widgets.HTML("<hr>"), widgets.Label("RATE FOOD"), rev_f, rev_s, btn_r])

        # TAB 3: PREDICTOR
        p_drop = widgets.Dropdown(options=food_options if food_options else ["No Foods"], description="Analyze:")
        p_btn = widgets.Button(description="Predict", button_style='warning')
        p_img = widgets.Image(width=200, height=200, layout={'display': 'none'})
        p_lab = widgets.Label()
        def pred(b):
            if p_drop.value == "No Foods": return
            f_obj = next(f for f in db['foods'] if f['name'] == p_drop.value)
            p_lab.value = f"📊 Result: {predict_score(f_obj, current_user)}"
            if f_obj.get('image'):
                try:
                    with open(os.path.join(IMG_FOLDER, f_obj['image']), 'rb') as f: p_img.value = f.read(); p_img.layout.display='block'
                except: p_img.layout.display='none'
            else: p_img.layout.display='none'
        p_btn.on_click(pred)
        tab3 = widgets.VBox([p_drop, p_btn, p_img, p_lab])

        # TAB 4: PROFILE
        psls = [widgets.IntSlider(value=current_user['traits'][i], min=-2, max=2, description=traits_list[i]) for i in range(7)]
        p_save = widgets.Button(description="Save Profile", button_style='primary')
        def save_p(b):
            current_user['traits'] = [s.value for s in psls]
            for u in db['users']:
                if u['username'] == current_user['username']: u['traits'] = current_user['traits']
            save_data(); print("💾 Saved!")
        p_save.on_click(save_p)
        tab4 = widgets.VBox([widgets.Label("YOUR PALATE SETTINGS")] + psls + [p_save])

        # ASSEMBLE TABS
        tabs_c = [tab1, tab2, tab3, tab4]; titles = ['Dashboard', 'Food Lab', 'Predictor', 'Profile']

        # TAB 5: ADMIN
        if is_admin:
            pend_drop = widgets.Dropdown(options=[f['name'] for f in db['pending_foods']] if db['pending_foods'] else ["No Pending"], description="Pending:")
            inspect_out = widgets.Output()

            def inspect_pending(change=None):
                with inspect_out:
                    clear_output()
                    if pend_drop.value == "No Pending": return
                    f_obj = next(f for f in db['pending_foods'] if f['name'] == pend_drop.value)
                    print(f"Submitted By: {f_obj.get('by', 'Unknown')}")
                    if f_obj.get('image'):
                        try:
                            with open(os.path.join(IMG_FOLDER, f_obj['image']), 'rb') as file:
                                display(widgets.Image(value=file.read(), width=150))
                        except: pass
                    for t, val in zip(traits_list, f_obj['traits']):
                        print(f"{t}: {val}")

            pend_drop.observe(inspect_pending, 'value')
            btn_app = widgets.Button(description="Approve", button_style='success')
            btn_rej = widgets.Button(description="Reject", button_style='danger')

            def app_f(b):
                if pend_drop.value == "No Pending": return
                f_obj = next(f for f in db['pending_foods'] if f['name'] == pend_drop.value)
                db['foods'].append(f_obj)
                db['pending_foods'] = [f for f in db['pending_foods'] if f['name'] != f_obj['name']]
                save_data(); refresh_ui()
            def rej_f(b):
                db['pending_foods'] = [f for f in db['pending_foods'] if f['name'] != pend_drop.value]
                save_data(); refresh_ui()
            btn_app.on_click(app_f); btn_rej.on_click(rej_f)

            u_del = widgets.Dropdown(options=[u['username'] for u in db['users'] if u['username']!="HenndralSaige"] or ["None"], description="User:")
            f_del = widgets.Dropdown(options=food_options or ["None"], description="Food:")
            btn_du = widgets.Button(description="Del User", button_style='danger')
            btn_df = widgets.Button(description="Del Food", button_style='danger')
            def d_u(b):
                if u_del.value != "None":
                    db['users'] = [u for u in db['users'] if u['username'] != u_del.value]
                    db['reviews'] = [r for r in db['reviews'] if r['user'] != u_del.value]
                    save_data(); refresh_ui()
            def d_f(b):
                if f_del.value != "None":
                    db['foods'] = [f for f in db['foods'] if f['name'] != f_del.value]
                    db['reviews'] = [r for r in db['reviews'] if r['food'] != f_del.value]
                    save_data(); refresh_ui()
            btn_du.on_click(d_u); btn_df.on_click(d_f)

            btn_wa_f = widgets.Button(description="WIPE ALL FOODS", button_style='danger')
            btn_wa_u = widgets.Button(description="WIPE ALL USERS", button_style='danger')
            conf_type = widgets.Label(value="")
            conf_btn = widgets.Button(description="CONFIRM DELETE?", button_style='danger', layout={'display': 'none'})
            def show_conf(b):
                conf_type.value = f"Ready to wipe {b.description.split('ALL ')[1]}"
                conf_btn.layout.display = 'block'
            def do_wipe(b):
                if "FOODS" in conf_type.value: db['foods'], db['reviews'], db['pending_foods'] = [], [], []
                else: db['users'] = [u for u in db['users'] if u['username']=="HenndralSaige"]; db['reviews'] = []
                save_data(); refresh_ui()
            btn_wa_f.on_click(show_conf); btn_wa_u.on_click(show_conf); conf_btn.on_click(do_wipe)

            tab_admin = widgets.VBox([
                widgets.Label("ADMIN APPROVAL QUEUE"), pend_drop, inspect_out, widgets.HBox([btn_app, btn_rej]), widgets.HTML("<hr>"),
                widgets.Label("SYSTEM MANAGEMENT"), u_del, btn_du, f_del, btn_df, widgets.HTML("<hr>"),
                widgets.Label("DANGER ZONE"), widgets.HBox([btn_wa_f, btn_wa_u]), conf_type, conf_btn
            ])
            tabs_c.append(tab_admin); titles.append('Admin')
            inspect_pending()

        t_ui = widgets.Tab(children=tabs_c)
        for i, t in enumerate(titles): t_ui.set_title(i, t)
        lo = widgets.Button(description="Logout")
        def out_a(b): global current_user; current_user = None; refresh_ui()
        lo.on_click(out_a)
        display(widgets.HTML(f"<h3>Welcome to FlavorMap, {current_user['username']}</h3>"), t_ui, lo)
    display(out)

refresh_ui()
